In [ ]:
import torch
import torch.nn as nn

from sklearn.metrics import confusion_matrix
import numpy as n

import rasterio as rio

In [ ]:
# Получение предсказаний по тестовой выборки

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in test_dataloader:
        X = val_tfms(batch['image']).to(device)
        y = remap_mask(batch['mask']).to(device)

        pred = model(X) #Для FCN: outputs = model(X)['out']
        pred = torch.argmax(pred, dim=1)

        all_preds.append(pred.cpu())
        all_targets.append(y.cpu())

preds = torch.cat(all_preds, dim=0).numpy()
targets = torch.cat(all_targets, dim=0).numpy()

In [ ]:
def metric(preds, targets, num_classes=num_classes): # Заменить на свое число классов
    mask = (targets != 255)
    flat_targets = targets[mask]
    flat_preds = preds[mask]
    
    cm = confusion_matrix(flat_targets.flatten(), flat_preds.flatten(), labels=range(num_classes))

    iou = np.diag(cm) / (cm.sum(1) + cm.sum(0) - np.diag(cm) + 1e-6)
    miou = iou.mean()
    
    # mAcc
    recall_per_class = np.diag(cm) / (cm.sum(1) + 1e-6)
    macc = recall_per_class.mean()
    
    
    # mF1
    f1_per_class = 2 * (precision_per_class * recall_per_class) / (precision_per_class + recall_per_class + 1e-6)
    mf1 = f1_per_class.mean()
    
    return miou, macc, mf1, iou, recall_per_class, f1_per_class

miou, macc, mf1, per_class_iou, recall_per_class, per_class_f1 = metric(preds, targets, num_classes=num_classes)

print(f"=== Результаты на тестовой выборке ===")
print(f"mIoU: {miou:.4f}")
print(f"mAcc (mean Recall): {macc:.4f}")
print(f"mF1: {mf1:.4f}")

print(f"\n--- По классам (0..{num_classes-1}) ---")
for i in range(num_classes):
    print(f"Класс {i}: IoU={per_class_iou[i]:.4f}, Recall={recall_per_class[i]:.4f}, F1={per_class_f1[i]:.4f}")

In [ ]:
def predict(model, image_path, save_path, patch_size=512, overlap=0.85):
    with rio.open(image_path) as src:
        full_img = src.read().astype(np.float32)
        profile = src.profile.copy()
        h, w = full_img.shape[1:]
    
    stride = int(patch_size * (1 - overlap))

    # Отражаем края изображения (при необходимости — для UNet)
    pad_size = patch_size // 2
    padded_img = np.pad(full_img, ((0, 0), (pad_size, pad_size), (pad_size, pad_size)), 
                        mode='reflect')
    padded_nodata = np.pad(nodata_mask, ((pad_size, pad_size), (pad_size, pad_size)), 
                           mode='reflect')
    padded_h, padded_w = padded_img.shape[1], padded_img.shape[2]
    
    # Определяем количество классов, которое предсказывает модель
    test_patch = padded_img[:, :patch_size, :patch_size]
    test_tensor = torch.tensor(test_patch).unsqueeze(0).float()
    test_norm = val_tfms(test_tensor).to(device)
    with torch.no_grad():
        n_classes = model(test_norm).shape[1]
    print(f"Model predicts {n_classes} classes")
    
    prob_sum = np.zeros((n_classes, padded_h, padded_w))
    count_map = np.zeros((padded_h, padded_w))
    
    for i in range(-pad_size, padded_h-patch_size+1, stride):
        for j in range(-pad_size, padded_w-patch_size+1, stride):
            i_start = max(0, i)
            j_start = max(0, j)
            i_end = min(padded_h, i + patch_size)
            j_end = min(padded_w, j + patch_size)
            
            patch = padded_img[:, i_start:i_end, j_start:j_end]
            current_h, current_w = patch.shape[1], patch.shape[2]
            
            if current_h != patch_size or current_w != patch_size:
                patch_full = np.zeros((patch.shape[0], patch_size, patch_size), dtype=np.float32)
                patch_full[:, :current_h, :current_w] = patch
            else:
                patch_full = patch
            
            patch_nodata = padded_nodata[i_start:i_end, j_start:j_end]
            if patch_nodata.sum() > patch_size*patch_size*0.9: # Если NoData превалирует
                continue
            
            patch_tensor = torch.tensor(patch_full).unsqueeze(0).float()
            patch_norm = val_tfms(patch_tensor).to(device)
            
            with torch.no_grad():
                pred = torch.softmax(model(patch_norm), dim=1)[0]
                if current_h != patch_size or current_w != patch_size:
                    pred = pred[:, :current_h, :current_w]
                
                prob_sum[:, i_start:i_end, j_start:j_end] += pred.cpu().numpy()
                count_map[i_start:i_end, j_start:j_end] += 1
    
    count_map = np.maximum(count_map, 1)
    prob_avg = (prob_sum / count_map)[:, pad_size:-pad_size, pad_size:-pad_size]
    
    full_mask = np.argmax(prob_avg, axis=0).astype(np.uint8)
    full_mask[nodata_mask] = 255
    
    profile.update({'count': 1, 'dtype': 'uint8', 'nodata': 255, 'compress': 'lzw'})
    with rio.open(save_path, 'w', **profile) as dst:
        dst.write(full_mask, 1)
    return full_mask


mask = predict(model, image_path='....tif', save_path='...tif')